In [1]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-AR"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 6538  Test: 727  PosTrain=278  PosTest=31
NR-AR RF AUC: 0.7307
NR-AR RF 分类报告:
               precision    recall  f1-score   support

         neg       0.97      1.00      0.98       696
      active       0.77      0.32      0.45        31

    accuracy                           0.97       727
   macro avg       0.87      0.66      0.72       727
weighted avg       0.96      0.97      0.96       727

NR-AR RF 混淆矩阵:
 [[693   3]
 [ 21  10]]
NR-AR SVM AUC: 0.7037
NR-AR SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.97      1.00      0.98       696
      active       0.80      0.39      0.52        31

    accuracy                           0.97       727
   macro avg       0.89      0.69      0.75       727
weighted avg       0.97      0.97      0.96       727

NR-AR SVM 混淆矩阵:
 [[693   3]
 [ 19  12]]
NR-AR XGB AUC: 0.7421
NR-AR XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.97      1.00      0.98     

In [4]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-AR-LBD"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 6082  Test: 676  PosTrain=213  PosTest=24
NR-AR-LBD RF AUC: 0.7782
NR-AR-LBD RF 分类报告:
               precision    recall  f1-score   support

         neg       0.97      0.99      0.98       652
      active       0.60      0.25      0.35        24

    accuracy                           0.97       676
   macro avg       0.79      0.62      0.67       676
weighted avg       0.96      0.97      0.96       676

NR-AR-LBD RF 混淆矩阵:
 [[648   4]
 [ 18   6]]
NR-AR-LBD SVM AUC: 0.8174
NR-AR-LBD SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.97      0.99      0.98       652
      active       0.64      0.29      0.40        24

    accuracy                           0.97       676
   macro avg       0.81      0.64      0.69       676
weighted avg       0.96      0.97      0.96       676

NR-AR-LBD SVM 混淆矩阵:
 [[648   4]
 [ 17   7]]
NR-AR-LBD XGB AUC: 0.7722
NR-AR-LBD XGB 分类报告:
               precision    recall  f1-score   support

         neg    

In [5]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-AhR"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5894  Test: 655  PosTrain=691  PosTest=77
NR-AhR RF AUC: 0.8899
NR-AhR RF 分类报告:
               precision    recall  f1-score   support

         neg       0.91      0.99      0.95       578
      active       0.85      0.22      0.35        77

    accuracy                           0.90       655
   macro avg       0.88      0.61      0.65       655
weighted avg       0.90      0.90      0.88       655

NR-AhR RF 混淆矩阵:
 [[575   3]
 [ 60  17]]
NR-AhR SVM AUC: 0.9004
NR-AhR SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.92      0.99      0.95       578
      active       0.76      0.34      0.47        77

    accuracy                           0.91       655
   macro avg       0.84      0.66      0.71       655
weighted avg       0.90      0.91      0.89       655

NR-AhR SVM 混淆矩阵:
 [[570   8]
 [ 51  26]]
NR-AhR XGB AUC: 0.8553
NR-AhR XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.92      0.97      0

In [6]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-Aromatase"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5238  Test: 583  PosTrain=270  PosTest=30
NR-Aromatase RF AUC: 0.8571
NR-Aromatase RF 分类报告:
               precision    recall  f1-score   support

         neg       0.96      1.00      0.98       553
      active       1.00      0.17      0.29        30

    accuracy                           0.96       583
   macro avg       0.98      0.58      0.63       583
weighted avg       0.96      0.96      0.94       583

NR-Aromatase RF 混淆矩阵:
 [[553   0]
 [ 25   5]]
NR-Aromatase SVM AUC: 0.8657
NR-Aromatase SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.96      1.00      0.98       553
      active       0.80      0.13      0.23        30

    accuracy                           0.95       583
   macro avg       0.88      0.57      0.60       583
weighted avg       0.95      0.95      0.94       583

NR-Aromatase SVM 混淆矩阵:
 [[552   1]
 [ 26   4]]
NR-Aromatase XGB AUC: 0.8336
NR-Aromatase XGB 分类报告:
               precision    recall  f1-score   s

In [7]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-ER"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5573  Test: 620  PosTrain=714  PosTest=79
NR-ER RF AUC: 0.6943
NR-ER RF 分类报告:
               precision    recall  f1-score   support

         neg       0.89      0.99      0.94       541
      active       0.77      0.13      0.22        79

    accuracy                           0.88       620
   macro avg       0.83      0.56      0.58       620
weighted avg       0.87      0.88      0.85       620

NR-ER RF 混淆矩阵:
 [[538   3]
 [ 69  10]]
NR-ER SVM AUC: 0.7087
NR-ER SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.89      1.00      0.94       541
      active       0.92      0.14      0.24        79

    accuracy                           0.89       620
   macro avg       0.90      0.57      0.59       620
weighted avg       0.89      0.89      0.85       620

NR-ER SVM 混淆矩阵:
 [[540   1]
 [ 68  11]]
NR-ER XGB AUC: 0.6901
NR-ER XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.90      0.98      0.94     

In [8]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-ER-LBD"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 6259  Test: 696  PosTrain=315  PosTest=35
NR-ER-LBD RF AUC: 0.7673
NR-ER-LBD RF 分类报告:
               precision    recall  f1-score   support

         neg       0.96      1.00      0.98       661
      active       0.67      0.17      0.27        35

    accuracy                           0.95       696
   macro avg       0.81      0.58      0.62       696
weighted avg       0.94      0.95      0.94       696

NR-ER-LBD RF 混淆矩阵:
 [[658   3]
 [ 29   6]]
NR-ER-LBD SVM AUC: 0.7901
NR-ER-LBD SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.96      1.00      0.98       661
      active       0.80      0.23      0.36        35

    accuracy                           0.96       696
   macro avg       0.88      0.61      0.67       696
weighted avg       0.95      0.96      0.95       696

NR-ER-LBD SVM 混淆矩阵:
 [[659   2]
 [ 27   8]]
NR-ER-LBD XGB AUC: 0.7593
NR-ER-LBD XGB 分类报告:
               precision    recall  f1-score   support

         neg    

In [9]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "NR-PPAR-gamma"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5805  Test: 645  PosTrain=167  PosTest=19
NR-PPAR-gamma RF AUC: 0.8826
NR-PPAR-gamma RF 分类报告:
               precision    recall  f1-score   support

         neg       0.97      1.00      0.99       626
      active       0.00      0.00      0.00        19

    accuracy                           0.97       645
   macro avg       0.49      0.50      0.49       645
weighted avg       0.94      0.97      0.96       645

NR-PPAR-gamma RF 混淆矩阵:
 [[626   0]
 [ 19   0]]
NR-PPAR-gamma SVM AUC: 0.8463
NR-PPAR-gamma SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.97      1.00      0.99       626
      active       0.00      0.00      0.00        19

    accuracy                           0.97       645
   macro avg       0.49      0.50      0.49       645
weighted avg       0.94      0.97      0.96       645

NR-PPAR-gamma SVM 混淆矩阵:
 [[626   0]
 [ 19   0]]
NR-PPAR-gamma XGB AUC: 0.8305
NR-PPAR-gamma XGB 分类报告:
               precision    recall  f1-s

In [10]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "SR-ARE"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5248  Test: 584  PosTrain=848  PosTest=94
SR-ARE RF AUC: 0.7871
SR-ARE RF 分类报告:
               precision    recall  f1-score   support

         neg       0.86      0.99      0.92       490
      active       0.71      0.13      0.22        94

    accuracy                           0.85       584
   macro avg       0.78      0.56      0.57       584
weighted avg       0.83      0.85      0.80       584

SR-ARE RF 混淆矩阵:
 [[485   5]
 [ 82  12]]
SR-ARE SVM AUC: 0.8318
SR-ARE SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.86      0.98      0.92       490
      active       0.69      0.19      0.30        94

    accuracy                           0.86       584
   macro avg       0.78      0.59      0.61       584
weighted avg       0.84      0.86      0.82       584

SR-ARE SVM 混淆矩阵:
 [[482   8]
 [ 76  18]]
SR-ARE XGB AUC: 0.8100
SR-ARE XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.88      0.98      0

In [11]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "SR-ATAD5"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 6364  Test: 708  PosTrain=238  PosTest=26
SR-ATAD5 RF AUC: 0.8616
SR-ATAD5 RF 分类报告:
               precision    recall  f1-score   support

         neg       0.96      1.00      0.98       682
      active       0.00      0.00      0.00        26

    accuracy                           0.96       708
   macro avg       0.48      0.50      0.49       708
weighted avg       0.93      0.96      0.95       708

SR-ATAD5 RF 混淆矩阵:
 [[682   0]
 [ 26   0]]
SR-ATAD5 SVM AUC: 0.8865
SR-ATAD5 SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.96      1.00      0.98       682
      active       0.00      0.00      0.00        26

    accuracy                           0.96       708
   macro avg       0.48      0.50      0.49       708
weighted avg       0.93      0.96      0.95       708

SR-ATAD5 SVM 混淆矩阵:
 [[682   0]
 [ 26   0]]
SR-ATAD5 XGB AUC: 0.8308
SR-ATAD5 XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.97 

In [12]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "SR-HSE"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5820  Test: 647  PosTrain=335  PosTest=37
SR-HSE RF AUC: 0.7485
SR-HSE RF 分类报告:
               precision    recall  f1-score   support

         neg       0.95      1.00      0.97       610
      active       0.50      0.05      0.10        37

    accuracy                           0.94       647
   macro avg       0.72      0.53      0.53       647
weighted avg       0.92      0.94      0.92       647

SR-HSE RF 混淆矩阵:
 [[608   2]
 [ 35   2]]
SR-HSE SVM AUC: 0.8405
SR-HSE SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.95      1.00      0.97       610
      active       1.00      0.05      0.10        37

    accuracy                           0.95       647
   macro avg       0.97      0.53      0.54       647
weighted avg       0.95      0.95      0.92       647

SR-HSE SVM 混淆矩阵:
 [[610   0]
 [ 35   2]]
SR-HSE XGB AUC: 0.7769
SR-HSE XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.95      1.00      0

In [13]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "SR-MMP"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 5229  Test: 581  PosTrain=826  PosTest=92
SR-MMP RF AUC: 0.8789
SR-MMP RF 分类报告:
               precision    recall  f1-score   support

         neg       0.87      0.99      0.93       489
      active       0.86      0.21      0.33        92

    accuracy                           0.87       581
   macro avg       0.87      0.60      0.63       581
weighted avg       0.87      0.87      0.83       581

SR-MMP RF 混淆矩阵:
 [[486   3]
 [ 73  19]]
SR-MMP SVM AUC: 0.9160
SR-MMP SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.89      0.99      0.94       489
      active       0.83      0.37      0.51        92

    accuracy                           0.89       581
   macro avg       0.86      0.68      0.72       581
weighted avg       0.88      0.89      0.87       581

SR-MMP SVM 混淆矩阵:
 [[482   7]
 [ 58  34]]
SR-MMP XGB AUC: 0.9196
SR-MMP XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.90      0.97      0

In [14]:
import pandas as pd, numpy as np, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, auc, 
                             classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix
import pandas as pd



warnings.filterwarnings("ignore")
os.makedirs("result-1", exist_ok=True)

# 读取数据
final_df = pd.read_csv("/Users/finn/Documents/research project/test36/final_df.csv")          
label = "SR-p53"
sub_df = final_df.dropna(subset=[label])        # dropna 
X = sub_df.iloc[:, 14:2446]                     # 特征 14:2445
y = sub_df[label].astype(int)
ids = sub_df['ID']

#  9:1 分层拆分
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.1, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}  "
      f"PosTrain={y_train.sum()}  PosTest={y_test.sum()}")

# 3. 模型池（默认）
models = {
    "RF": RandomForestClassifier(n_jobs=-1, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "XGB": XGBClassifier(n_jobs=-1, random_state=42, eval_metric='logloss'),
    "NB": GaussianNB()
}

# 4. 逐模型跑 & 保存
plt.figure(figsize=(4, 4))
metrics_data = []
for mname, model in models.items():
    # 训练
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc_score = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['neg', 'active'])
    precision = report['active']['precision']
    recall = report['active']['recall']
    f1 = report['active']['f1-score']
    metrics_data.append([mname, f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}', f'{auc_score:.4f}'])

    # 评估
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"{label} {mname} AUC: {auc_score:.4f}")
    print(f"{label} {mname} 分类报告:\n", classification_report(y_test, y_pred, target_names=['neg', 'active']))
    print(f"{label} {mname} 混淆矩阵:\n", confusion_matrix(y_test, y_pred))
    
    # 评估报告 DataFrame
    report_df = pd.DataFrame({
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-score': f'{f1:.4f}',
        'AUC': f'{auc_score:.4f}'
    }, index=[mname], dtype=float)  # 指定为浮点数

    report_df.to_csv(f"result-1/{label}_{mname}_Classification_Report.csv", index=False)

    # 原始 ROC 数据
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
        f"result-1/{label}_{mname}_原始ROC数据.csv", index=False)

    # ROC 曲线
    
    plt.plot(fpr, tpr, label=f'{mname} AUC={auc_score:.3f}')




# ===== 5. 图保存 =====
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC - {label}'); plt.legend(); plt.tight_layout()
plt.savefig(f"result-1/{label}_ROC.png", dpi=300); plt.close()

print(f"{label} roc完成")

# ===== 6. 性能指标表格 ===== 【在这里添加】
from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('tight')
ax.axis('off')
columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'AUC']
table = ax.table(cellText=metrics_data, colLabels=columns, cellLoc='center', loc='center', colWidths=[0.15, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)
for i in range(len(columns)):
    table[(0, i)].set_facecolor("#7f5a9d78")
    table[(0, i)].set_text_props(weight='bold', color='white')
colors = ['#f0f0f0', '#ffffff']
for i in range(1, len(metrics_data) + 1):
    for j in range(len(columns)):
        table[(i, j)].set_facecolor(colors[i % 2])
plt.title(f'Classification Report - {label}', fontsize=14, weight='bold', pad=20)
plt.savefig(f"result-1/{label}_Classification_Report.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"{label} 表格完成")

Train: 6096  Test: 678  PosTrain=381  PosTest=42
SR-p53 RF AUC: 0.8099
SR-p53 RF 分类报告:
               precision    recall  f1-score   support

         neg       0.94      1.00      0.97       636
      active       0.50      0.02      0.05        42

    accuracy                           0.94       678
   macro avg       0.72      0.51      0.51       678
weighted avg       0.91      0.94      0.91       678

SR-p53 RF 混淆矩阵:
 [[635   1]
 [ 41   1]]
SR-p53 SVM AUC: 0.8358
SR-p53 SVM 分类报告:
               precision    recall  f1-score   support

         neg       0.94      1.00      0.97       636
      active       0.50      0.02      0.05        42

    accuracy                           0.94       678
   macro avg       0.72      0.51      0.51       678
weighted avg       0.91      0.94      0.91       678

SR-p53 SVM 混淆矩阵:
 [[635   1]
 [ 41   1]]
SR-p53 XGB AUC: 0.8756
SR-p53 XGB 分类报告:
               precision    recall  f1-score   support

         neg       0.94      0.99      0

In [ ]:
import os
import pandas as pd
import glob

# 获取所有 Classification_Report.csv 文件
result-1_dir = "result-1"
csv_files = glob.glob(os.path.join(result-1_dir, "*_Classification_Report.csv"))

# 汇总所有数据
all_data = []

for csv_file in sorted(csv_files):
    # 读取 CSV 文件
    df = pd.read_csv(csv_file)
    
    # 从文件名提取 label 和 model 信息
    filename = os.path.basename(csv_file)
    # 格式: {label}_{model}_Classification_Report.csv
    parts = filename.replace("_Classification_Report.csv", "").rsplit("_", 1)
    label = parts[0]
    model = parts[1] if len(parts) > 1 else "Unknown"
    
    # 添加 Label 和 Model 列
    df.insert(0, 'Model', model)
    df.insert(0, 'Label', label)
    
    all_data.append(df)

# 合并所有数据
combined_df = pd.concat(all_data, ignore_index=True)

# 保存合并后的 CSV
combined_df.to_csv(f"{result-1_dir}/All_Classification_Report.csv", index=False)

print("所有分类报告已合并保存为：All_Classification_Report.csv")
print(combined_df)


所有分类报告已合并保存为：All_Classification_Report.csv
            Label Model  Precision  Recall  F1-score     AUC
0       NR-AR-LBD    NB     0.1163  0.4167    0.1818  0.6501
1       NR-AR-LBD    RF     0.6000  0.2500    0.3529  0.7782
2       NR-AR-LBD   SVM     0.6364  0.2917    0.4000  0.8174
3       NR-AR-LBD   XGB     0.7273  0.3333    0.4571  0.7722
4           NR-AR    NB     0.0929  0.5484    0.1589  0.6549
5           NR-AR    RF     0.7692  0.3226    0.4545  0.7307
6           NR-AR   SVM     0.8000  0.3871    0.5217  0.7037
7           NR-AR   XGB     0.7857  0.3548    0.4889  0.7421
8          NR-AhR    NB     0.1495  0.7143    0.2472  0.5882
9          NR-AhR    RF     0.8500  0.2208    0.3505  0.8899
10         NR-AhR   SVM     0.7647  0.3377    0.4685  0.9004
11         NR-AhR   XGB     0.6190  0.3377    0.4370  0.8553
12   NR-Aromatase    NB     0.0548  0.2667    0.0909  0.5086
13   NR-Aromatase    RF     1.0000  0.1667    0.2857  0.8571
14   NR-Aromatase   SVM     0.8000  0.1333

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# 设置字体为 Times New Roman
rcParams['font.family'] = 'Times New Roman'

# 读取合并后的 CSV
df = pd.read_csv("result-1/All_Classification_Report.csv")

# 提取数值列用于热力图
numeric_cols = ['Precision', 'Recall', 'F1-score', 'AUC']
data_for_heatmap = df[numeric_cols].values

# 创建行标签（Label + Model）
row_labels = df['Label'] + ' - ' + df['Model']

# 创建热力图
fig, ax = plt.subplots(figsize=(10, 16))
sns.heatmap(data_for_heatmap, 
            annot=True,  # 显示数值
            fmt='.4f',   # 格式：4位小数
            cmap='Blues',  # 红-黄-绿配色（红=低，绿=高）
            cbar_kws={'label': 'Score'},
            xticklabels=numeric_cols,
            yticklabels=row_labels,
            ax=ax,
            vmin=0,  # 最小值 0
            vmax=1,  # 最大值 1
            linewidths=0.5,
            linecolor='gray')

# 设置标题和标签
plt.title('Classification Report - Heatmap', fontsize=14, weight='bold', pad=20)
plt.xlabel('Metrics', fontsize=12)
plt.ylabel('Label - Model', fontsize=12)
plt.tight_layout()

# 保存图片
plt.savefig("result-1/All_Classification_Report_Heatmap.png", dpi=300, bbox_inches='tight')
plt.close()

print("热力图已保存为：All_Classification_Report_Heatmap.png")


热力图已保存为：All_Classification_Report_Heatmap.png


In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# 设置字体
rcParams['font.family'] = 'Times New Roman'

# 读取数据
df = pd.read_csv("result-1/All_Classification_Report.csv")

numeric_cols = ['Recall', 'Recall', 'F1-score', 'AUC']

# ============= 按 Label 分组 =============
labels = df['Label'].unique()
n_labels = len(labels)

fig, axes = plt.subplots(n_labels, 1, figsize=(8, 3*n_labels))
if n_labels == 1:
    axes = [axes]

for idx, label in enumerate(sorted(labels)):
    label_data = df[df['Label'] == label][numeric_cols].values
    models = df[df['Label'] == label]['Model'].values
    
    sns.heatmap(label_data,
                annot=True,
                fmt='.4f',
                cmap='Blues',
                cbar_kws={'label': 'Score'},
                xticklabels=numeric_cols,
                yticklabels=models,
                ax=axes[idx],
                vmin=0,
                vmax=1,
                linewidths=0.5,
                linecolor='gray')
    axes[idx].set_title(f'{label}', fontsize=12, weight='bold')
    axes[idx].set_ylabel('Model', fontsize=10)

plt.tight_layout()
plt.savefig("result-1/Classification_Report_By_Label.png", dpi=300, bbox_inches='tight')
plt.close()

print("按 Label 分组的热力图已保存为：Classification_Report_By_Label.png")

# ============= 按 Model 分组 =============
models = df['Model'].unique()
n_models = len(models)

fig, axes = plt.subplots(n_models, 1, figsize=(8, 3*n_models))
if n_models == 1:
    axes = [axes]

for idx, model in enumerate(sorted(models)):
    model_data = df[df['Model'] == model][numeric_cols].values
    labels_list = df[df['Model'] == model]['Label'].values
    
    sns.heatmap(model_data,
                annot=True,
                fmt='.4f',
                cmap='Blues',
                cbar_kws={'label': 'Score'},
                xticklabels=numeric_cols,
                yticklabels=labels_list,
                ax=axes[idx],
                vmin=0,
                vmax=1,
                linewidths=0.5,
                linecolor='gray')
    axes[idx].set_title(f'{model}', fontsize=12, weight='bold')
    axes[idx].set_ylabel('Label', fontsize=10)

plt.tight_layout()
plt.savefig("result-1/Classification_Report_By_Model.png", dpi=300, bbox_inches='tight')
plt.close()

print("按 Model 分组的热力图已保存为：Classification_Report_By_Model.png")


按 Label 分组的热力图已保存为：Classification_Report_By_Label.png
按 Model 分组的热力图已保存为：Classification_Report_By_Model.png


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# 设置字体
rcParams['font.family'] = 'Times New Roman'

# 读取数据
df = pd.read_csv("result-1/All_Classification_Report.csv")

# 创建透视表：行为 Model，列为 Label，值为 Precision
pivot_df = df.pivot_table(index='Model', columns='Label', values='Precision')

# 按照需要的顺序排列（可选）
pivot_df = pivot_df.reindex(['NB', 'RF', 'SVM', 'XGB'])  # Model 顺序
pivot_df = pivot_df[sorted(pivot_df.columns)]  # Label 按字母排序

# 创建热力图
fig, ax = plt.subplots(figsize=(12, 4))

sns.heatmap(pivot_df,
            annot=True,
            fmt='.2f',
            cmap='Blues',
            cbar_kws={'label': 'Precision'},
            ax=ax,
            vmin=0,
            vmax=1,
            linewidths=0.5,
            linecolor='gray',
            cbar=True,
            annot_kws={'size': 13})

# 调整坐标轴标签字体大小
ax.set_xticklabels(ax.get_xticklabels(), fontsize=13)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=13)

# 设置标题和标签
plt.title('Precision Heatmap (Model × Label)', fontsize=14, weight='bold', pad=20)
plt.xlabel('Label', fontsize=13)
plt.ylabel('Model', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# 调整 colorbar 标签字体大小
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=13)
cbar.set_label('Precision', fontsize=13)

# 保存图片
plt.savefig("result-1-1/Precision_Heatmap_Model_Label.png", dpi=300, bbox_inches='tight')
plt.close()

print("Model × Label 的 Precision 热力图已保存为：Precision_Heatmap_Model_Label.png")
print("热力图矩阵：")
print(pivot_df)


Model × Label 的 Precision 热力图已保存为：Precision_Heatmap_Model_Label.png
热力图矩阵：
Label   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase   NR-ER  NR-ER-LBD  \
Model                                                               
NB     0.0929     0.1163  0.1495        0.0548  0.1307     0.0899   
RF     0.7692     0.6000  0.8500        1.0000  0.7692     0.6667   
SVM    0.8000     0.6364  0.7647        0.8000  0.9167     0.8000   
XGB    0.7857     0.7273  0.6190        1.0000  0.6296     0.7000   

Label  NR-PPAR-gamma  SR-ARE  SR-ATAD5  SR-HSE  SR-MMP  SR-p53  
Model                                                           
NB            0.0533  0.1812    0.0816  0.0588  0.1734  0.0581  
RF            0.0000  0.7059    0.0000  0.5000  0.8636  0.5000  
SVM           0.0000  0.6923    0.0000  1.0000  0.8293  0.5000  
XGB           0.8000  0.7250    0.5714  0.7143  0.7636  0.4545  


In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# 设置字体
rcParams['font.family'] = 'Times New Roman'

# 读取数据
df = pd.read_csv("result-1/All_Classification_Report.csv")

# 创建透视表：行为 Model，列为 Label，值为 Recall
pivot_df = df.pivot_table(index='Model', columns='Label', values='Recall')

# 按照需要的顺序排列（可选）
pivot_df = pivot_df.reindex(['NB', 'RF', 'SVM', 'XGB'])  # Model 顺序
pivot_df = pivot_df[sorted(pivot_df.columns)]  # Label 按字母排序

# 创建热力图
fig, ax = plt.subplots(figsize=(12, 4))

sns.heatmap(pivot_df,
            annot=True,
            fmt='.2f',
            cmap='Blues',
            cbar_kws={'label': 'Recall'},
            ax=ax,
            vmin=0,
            vmax=1,
            linewidths=0.5,
            linecolor='gray',
            cbar=True,
            annot_kws={'size': 13})

# 调整坐标轴标签字体大小
ax.set_xticklabels(ax.get_xticklabels(), fontsize=13)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=13)

# 设置标题和标签
plt.title('Recall Heatmap (Model × Label)', fontsize=14, weight='bold', pad=20)
plt.xlabel('Label', fontsize=13)
plt.ylabel('Model', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# 调整 colorbar 标签字体大小
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=13)
cbar.set_label('Recall', fontsize=13)

# 保存图片
plt.savefig("result-1/Recall_Heatmap_Model_Label.png", dpi=300, bbox_inches='tight')
plt.close()

print("Model × Label 的 Recall 热力图已保存为：Recall_Heatmap_Model_Label.png")
print("热力图矩阵：")
print(pivot_df)


Model × Label 的 Recall 热力图已保存为：Recall_Heatmap_Model_Label.png
热力图矩阵：
Label   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase   NR-ER  NR-ER-LBD  \
Model                                                               
NB     0.5484     0.4167  0.7143        0.2667  0.6582     0.4571   
RF     0.3226     0.2500  0.2208        0.1667  0.1266     0.1714   
SVM    0.3871     0.2917  0.3377        0.1333  0.1392     0.2286   
XGB    0.3548     0.3333  0.3377        0.1667  0.2152     0.2000   

Label  NR-PPAR-gamma  SR-ARE  SR-ATAD5  SR-HSE  SR-MMP  SR-p53  
Model                                                           
NB            0.2105  0.8830    0.4615  0.3514  0.7500  0.4524  
RF            0.0000  0.1277    0.0000  0.0541  0.2065  0.0238  
SVM           0.0000  0.1915    0.0000  0.0541  0.3696  0.0238  
XGB           0.2105  0.3085    0.1538  0.1351  0.4565  0.1190  
